# 31 - Higher Homotopy Groups $\pi_n$: Hurewicz, Whitehead Towers & Adams Integration

The fundamental group $\pi_1$ classifies covering spaces. The higher homotopy groups $\pi_n$ for $n \geq 2$ classify higher-dimensional sphere attachments in the Postnikov tower. Computing them is *hard*: $\pi_n(S^2)$ contains torsion of arbitrarily high order.

`pysurgery.core.higher_homotopy_groups` combines three sources of data to give the best possible answer:
1. **Sullivan model** (NB 26): gives $\pi_n \otimes \mathbb{Q}$ exactly
2. **Adams spectral sequence** (NB 23): gives the $p$-primary torsion via $\mathrm{Ext}_{\mathcal{A}}$
3. **E∞ resolver** (`pysurgery.core.e_infinity_resolver`): resolves extension ambiguities

The key function `synthesize_homotopy_group_with_e_infinity` orchestrates all three.

## Learning Goals
- Understand the Hurewicz homomorphism and when $\pi_n \cong H_n$
- Understand Whitehead towers and Postnikov sections
- Use `synthesize_homotopy_group_with_e_infinity` for the best combined estimate
- Read `HomotopyGroupApproximation.confidence` and understand uncertainty
- Understand when `exact=True` is achievable vs. when confidence is < 1

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Space $X$ | `SimplicialComplex` |
| **Algebraic** | $\pi_n(X)$ as abelian group (for $n \geq 2$) | `synthesize_homotopy_group_with_e_infinity` |
| **Result** | `HomotopyGroupApproximation`: group + confidence score | `.group_description`, `.confidence` |

## Formal Background

**Hurewicz theorem**: If $X$ is $(n-1)$-connected (i.e. $\pi_k(X) = 0$ for $k < n$), then the Hurewicz homomorphism $h: \pi_n(X) \to H_n(X)$ is an isomorphism.

**Whitehead tower**: $\cdots \to X\langle n \rangle \to X\langle n-1 \rangle \to \cdots \to X$ where $X\langle n \rangle$ is the $n$-connected cover. The fiber of $X\langle n \rangle \to X\langle n-1 \rangle$ is $K(\pi_n(X), n)$ (an Eilenberg-MacLane space).

In [ ]:
from pysurgery.core.higher_homotopy_groups import (
    HomotopyGroupApproximation,
    synthesize_homotopy_group_with_e_infinity,
    rational_homotopy_group,
)
from pysurgery.core.complexes import SimplicialComplex

print("=" * 70)
print("31 - Higher Homotopy Groups: Setup Complete")
print("=" * 70)

## Part 1: The Hurewicz Isomorphism

For simply-connected spaces, the first non-trivial $\pi_n$ equals the first non-trivial $H_n$ (Hurewicz theorem). This gives a quick lower bound: if $H_3(X) \neq 0$ and $H_k(X) = 0$ for $k = 1, 2$, then $\pi_3(X) \cong H_3(X)$ (plus possible torsion from higher Steenrod operations).

In [ ]:
# Hurewicz cases: simply-connected spaces where π_n ≅ H_n
spaces = {
    "S²": SimplicialComplex.sphere(2),
    "S³": SimplicialComplex.sphere(3),
    "CP²": SimplicialComplex.complex_projective_space(2),
}

print("Hurewicz isomorphism check (first non-trivial degree):")
for name, X in spaces.items():
    betti = [X.betti_number(k) for k in range(5)]
    first_nontriv = next((k for k, b in enumerate(betti) if b > 0 and k > 0), None)
    print(f"  {name}: H_{first_nontriv} = Z^{betti[first_nontriv]} → π_{first_nontriv} ⊇ Z^{betti[first_nontriv]} (Hurewicz)")

## Part 2: Rational Part from Sullivan Models

For simply-connected spaces, the rational homotopy groups are read directly from the indecomposables of the Sullivan minimal model. This computation is always exact.

In [ ]:
from pysurgery.core.rational_homotopy import (
    sphere_cohomology, complex_projective_space_cohomology,
    rational_homotopy_group,
)

print("Rational homotopy groups π_n ⊗ Q:")
print()

for name, H in [("S²", sphere_cohomology(2)),
                ("S³", sphere_cohomology(3)),
                ("CP²", complex_projective_space_cohomology(2))]:
    print(f"{name}:")
    nonzero = []
    for n in range(2, 10):
        pi_n = rational_homotopy_group(H, n=n)
        if pi_n.rank > 0:
            nonzero.append((n, pi_n.rank))
            print(f"  π_{n} ⊗ Q = Q^{pi_n.rank}  (exact={pi_n.exact})")
    if not nonzero:
        print("  All rational homotopy groups trivial in range 2-9")
    print()

print("Note: S³ has only π₃⊗Q=Q — it is rationally a K(Z,3) space")
print("      CP² has π₂⊗Q=Q and π₅⊗Q=Q (generator + Whitehead product)")

## Part 3: Full Synthesis with Adams SS + E∞ Resolver

`synthesize_homotopy_group_with_e_infinity` is the main function. It:
1. Calls `rational_homotopy_group` for the $\mathbb{Q}$-part (rank)
2. Calls `adams_e2_page` to get $\mathrm{Ext}^{s,t}_{\mathcal{A}}$ for the $p$-primary torsion
3. Runs the Adams SS through convergence for each prime $p$ in `primes`
4. Calls the E∞ resolver for extension ambiguities
5. Assembles a `HomotopyGroupApproximation` with a `confidence` score

The `confidence` is $\in [0, 1]$: 1.0 means certified, < 1 means some steps were heuristic (e.g., Adams differentials above page 2 were not computed).

In [ ]:
# π₄(S³) = Z/2 — the classic first non-trivial homotopy group
S3 = SimplicialComplex.sphere(3)

approx: HomotopyGroupApproximation = synthesize_homotopy_group_with_e_infinity(
    space=S3,
    n=4,
    prime=2,
    use_adams=True,
    use_sullivan=True,
)

print("π₄(S³) synthesis:")
print(f"  group_description: {approx.group_description}")
print(f"  rank:              {approx.rank}")
print(f"  torsion_orders:    {approx.torsion_orders}")
print(f"  confidence:        {approx.confidence:.2f}")
print(f"  sources:           {approx.sources}")
print(f"  exact:             {approx.exact}")
print(f"  theorem_tag:       {approx.theorem_tag}")
print()
print("Known result: π₄(S³) = Z/2  ← Hopf invariant one")
assert approx.torsion_orders == [2], f"Expected [2], got {approx.torsion_orders}"
print("Assertion passed: torsion_orders == [2] ✓")

In [ ]:
# Compute a range of homotopy groups for S²
print("Homotopy groups of S² (first 10):")
print(f"{'n':>4} {'group':>20} {'confidence':>12} {'exact':>7}")
print("-" * 50)

for n in range(2, 11):
    try:
        a = synthesize_homotopy_group_with_e_infinity(
            space=SimplicialComplex.sphere(2), n=n, prime=2,
            use_adams=True, use_sullivan=True,
        )
        grp = a.group_description or f"Z^{a.rank}"
        if a.torsion_orders:
            grp += " ⊕ " + " ⊕ ".join(f"Z/{t}" for t in a.torsion_orders)
        print(f"{n:>4}  {grp:>20}  {a.confidence:>12.2f}  {str(a.exact):>7}")
    except Exception as e:
        print(f"{n:>4}  {'(error)':>20}  {'—':>12}  {'—':>7}  [{e}]")

## Part 4: Confidence Scores and Uncertainty

The confidence score reflects which steps were provably correct:
- **Sullivan model**: always exact over $\mathbb{Q}$ → full credit for rational part
- **Adams $E_2$ page**: exact (minimal resolution is algorithmic) → full credit
- **Adams differentials $d_r, r \geq 3$**: currently heuristic → confidence < 1
- **E∞ extension resolution**: exact when secondary ops are known → credit if resolved

Confidence < 1 means: the result is the best available estimate, but a full proof requires checking higher Adams differentials by hand.

In [ ]:
# Confidence breakdown
for n in [4, 5, 6, 7]:
    a = synthesize_homotopy_group_with_e_infinity(
        space=SimplicialComplex.sphere(3), n=n, prime=2,
        use_adams=True, use_sullivan=True,
    )
    print(f"π_{n}(S³): {a.group_description or 0}  "
          f"confidence={a.confidence:.2f}  sources={a.sources}")

print()
print("Low confidence (< 0.8) means higher Adams differentials were not checked.")
print("Use the Adams SS notebook (NB 23) to verify d_r for r ≥ 3 manually.")

## Summary Checklist

- [x] Applied Hurewicz theorem to identify first non-trivial $\pi_n$ for simply-connected spaces
- [x] Computed $\pi_n \otimes \mathbb{Q}$ via Sullivan models
- [x] Used `synthesize_homotopy_group_with_e_infinity` for the full combined estimate
- [x] Read `HomotopyGroupApproximation`: group description, rank, torsion, confidence
- [x] Understood when `exact=True` vs. `confidence < 1`

## Next Steps
- **NB 23**: Spectral sequences — use Adams SS directly if you need higher differentials
- **NB 26**: Rational homotopy — deepens the Sullivan model computation
- **NB 29**: Rational surgery uses $\pi_n$ data to compute $L$-group obstructions